Script: loads the SEOF, regresses on 500mb geopotential height, then plots panels - similar to O'Brien & Deser Figure S1

This is for obs

Output files:
FS1bdata_zgTPac

In [2]:
#load packages needed in this notebook
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import os
from eofs.standard import Eof
import regionmask
import cartopy.crs as ccrs
from matplotlib import pyplot as PP
import copy
import matplotlib as mpl
import matplotlib.patches as mpatches
from cartopy.util import add_cyclic_point

In [3]:
#define linear regression function

def linregress_3D(x, y):
    """

    Input:
    x: one dimentional array to regress against
    y: 3 dimentionsal array (time,lat,lon)

    the first dimention of y must be the same length of x

    return: slope,intercept,pval,cor,stderr,cov on regression
    between the two datasets along their aligned dimension.

    The output format and data are exactly identical to numpy's linregress

    """

    # broadcast the input x array to be the same shape as input y array
    x = np.array(np.broadcast_to(x[:,np.newaxis,np.newaxis],y.shape))

    # mask invalid locations in y
    ym = np.ma.masked_invalid(y)
    # apply the same mask to x
    xm = np.ma.masked_array(x, mask=ym.mask)

    #3. Compute data length, mean and standard deviation along time axis for further use:
    n     = xm.shape[0]
    xmean = xm.mean(axis=0)
    ymean = ym.mean(axis=0)
    xstd  = xm.std(axis=0)
    ystd  = ym.std(axis=0)

    #4. Compute covariance along time axis
    cov   =  np.sum(np.multiply((xm - xmean),(ym - ymean)), axis=0)/(n-1)
    #covxx   =  np.sum(np.multiply((xm - xmean),(xm - xmean).T), axis=0)/(n-1)
    #covyy   =  np.sum(np.multiply((ym - ymean),(ym - ymean).T), axis=0)/(n-1)
    #print(cov.shape)

    #5. Compute correlation along time axis
    cor   = cov/(xstd*ystd)

    #6. Compute regression slope and intercept:
    slope     = cov/(xstd**2)
    intercept = ymean - xmean*slope

    #7. Compute P-value and standard error
    # Compute t-statistics
    TINY = 1.0e-20
    tstats = cor*np.sqrt((n-2)/((1.0 - cor + TINY)*(1.0 + cor + TINY)))
    stderr = slope/tstats

    df = n-2
    #stderr = np.sqrt((1 - cor**2) * covxx / covyy / df)

    from scipy.stats import t
    pval   = 2*t.sf(x=np.abs(tstats), df=df)
    #pval   = xr.DataArray(pval, dims=cor.dims, coords=cor.coords)

    return slope, intercept, cor, pval, stderr

In [4]:
#set up the data directory and load in lon + lat + time dimensions
outputdir='/glade/campaign/cgd/cas/nmaher/cesm1_lens/Amon/zg/' 
outputdir2='/glade/work/nmaher/SEOF_output/'
model='OBS'


#load lon lat should be same for all models
ds_fx = xr.open_dataset(outputdir+'zg_Amon_CESM1-CAM5_historical_rcp85_r10i1p1_192001-210012_g025.nc')

lon = ds_fx.lon
lat = ds_fx.lat

In [5]:
#load data from PNA SEOF and from the saved DJF zg500 files

eof_T='_TPAC_SST'

with np.load(outputdir2+model+eof_T+'_SPCS.npz') as npz:
    pna_spcs = np.ma.MaskedArray(**npz)
with np.load(outputdir2+model+eof_T+'_SEOF.npz') as npz:
    pna_seof = np.ma.MaskedArray(**npz)

eof_T='_zgDJFerem'
with np.load(outputdir2+model+eof_T+'_SEOF.npz') as npz:
    zg_DJF_2e_full = np.ma.MaskedArray(**npz)
    
eof_T='_zgDJF'
with np.load(outputdir2+model+eof_T+'_SEOF.npz') as npz:
    zg_DJF_2_full = np.ma.MaskedArray(**npz)    

In [6]:
#tell me how many eofs and length of time dimension
neof=2


In [7]:


pna_seof_tmp = np.ma.masked_array(pna_seof,mask=pna_seof.mask,copy=True)
pna_seof_phcorr = np.ma.masked_array(pna_seof,mask=pna_seof.mask,copy=True)

pna_spcs_tmp = np.array(pna_spcs,copy=True)
pna_spcs_phcorr = np.array(pna_spcs,copy=True)


In [8]:
# now regess the phase corrected seof principal components against the z500 ensemble dimension

z500_spc_ens_reg_slopes = np.ma.masked_equal(np.zeros((neof,72,144)),0)
z500_spc_ens_reg_pvals = np.ma.masked_equal(np.zeros((neof,72,144)),0)
z500_spc_ens_reg_corrcoef = np.ma.masked_equal(np.zeros((neof,72,144)),0)

for j in range(neof):


    z500_rs = zg_DJF_2e_full.reshape(-1,72,144)

    z_ens_pc_reg = linregress_3D(pna_spcs_phcorr[:,j], z500_rs)
    z500_spc_ens_reg_slopes[j,...] = z_ens_pc_reg[0]
    z500_spc_ens_reg_pvals[j,...] = z_ens_pc_reg[3]
    z500_spc_ens_reg_corrcoef[j,...] = z_ens_pc_reg[2]

In [9]:
#other models load in CESM1 - correlate and decide to flip

model2='CESM1-LE'

with np.load(outputdir2+model2+'_tpac_reg_slope.npz') as npz:
    pna_cesm = np.ma.MaskedArray(**npz)   
    



In [11]:
#do the flipping to match CESM1

eof0a = pna_cesm[0,0,...].flatten() 
eof1a = z500_spc_ens_reg_slopes[0,:,:].flatten() 
num = np.sum(eof0a*eof1a)/np.sqrt(np.sum(eof0a**2)*np.sum(eof1a**2))
if num<0:
    z500_spc_ens_reg_slopes[0,:,:]=z500_spc_ens_reg_slopes[0,:,:]*-1
    
eof0a = pna_cesm[1,0,...].flatten() 
eof1a = z500_spc_ens_reg_slopes[1,:,:].flatten() 
num = np.sum(eof0a*eof1a)/np.sqrt(np.sum(eof0a**2)*np.sum(eof1a**2))
if num<0:
    z500_spc_ens_reg_slopes[1,:,:]=z500_spc_ens_reg_slopes[1,:,:]*-1
    


In [12]:
#save the data that goes into the Figure for easy access
outputdir_figD='/glade/work/nmaher/SEOF_output/Fig_data/'

eof_T='_FS1bdata_zgTPac'
np.savez_compressed(outputdir_figD+model+eof_T+'.npz', data=z500_spc_ens_reg_slopes.data)
